#MINI PROJECT

In [2]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [5]:
# ==========================================================
# MINI PROJECT : COLLEGE KNOWLEDGE ASSISTANT
# ==========================================================

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# ----------------------------------------------------------
# 1. Load Knowledge Base
# ----------------------------------------------------------

df = pd.read_csv("college_notes.csv")

print(f"Loaded {len(df)} notes")

# ----------------------------------------------------------
# 2. Initialize Embedding Model
# ----------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------------------------------------
# 3. Create ChromaDB Collection
# ----------------------------------------------------------

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="college_notes"
)

# ----------------------------------------------------------
# 4. Store Notes with Embeddings
# ----------------------------------------------------------

documents = []
metadatas = []
ids = []

for i, row in df.iterrows():

    text = str(row["content"])

    documents.append(text)

    metadatas.append({
        "subject": row["subject"],
        "topic": row["topic"]
    })

    ids.append(str(i))

embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("Knowledge Base Indexed Successfully!")

# ----------------------------------------------------------
# 5. Initialize Groq
# ----------------------------------------------------------

groq_client = Groq(
    api_key="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"
)

# ----------------------------------------------------------
# 6. RAG Function
# ----------------------------------------------------------

def ask_college_assistant(question, top_k=3):

    # Retrieve Relevant Notes
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # Handle Out-of-Scope Questions
    if len(docs) == 0:
        return "Sorry, this information is not available in the college knowledge base."

    # Build Context
    context = "\n\n".join(docs)

    prompt = f"""
You are a College Knowledge Assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say:
'Sorry, this information is not available in the college knowledge base.'
"""

    # Generate Answer
    response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    answer = response.choices[0].message.content

    # Add Citations
    citations = "\n\nSources:\n"

    for meta in metas:
        citations += f"- {meta['subject']} : {meta['topic']}\n"

    return answer + citations

# ----------------------------------------------------------
# 7. Ask Question
# ----------------------------------------------------------

question = input("Ask a Question: ")

answer = ask_college_assistant(question)

print("\nAnswer:\n")
print(answer)

Loaded 15 notes


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge Base Indexed Successfully!
Ask a Question: N004 subject name

Answer:

Sorry, this information is not available in the college knowledge base.

Sources:
- Generative AI : Large Language Models
- Generative AI : Retrieval Augmented Generation
- Generative AI : Prompt Engineering

